# Chain-of-Thought Distillation: SmolLM2-360M on GSM8K

Distill Claude Haiku 4.5's chain-of-thought reasoning into SmolLM2-360M-Instruct via SFT-LoRA, then measure accuracy and reasoning quality on GSM8K.

## Pipeline

1. Generate 1500 CoT solutions to GSM8K training problems with Claude Haiku 4.5.
2. Reject-sample: keep only CoTs whose final answer matches gold (~95% acceptance).
3. SFT-LoRA fine-tune SmolLM2-360M-Instruct on the filtered (question, CoT) pairs.
4. Evaluate baseline and fine-tuned models on 200 held-out problems.
5. Bootstrap confidence intervals on the per-problem difference.

Compute: single Kaggle T4 (free tier), ~$3 in Anthropic API costs.

The base model is small enough that distillation can transfer reasoning *style* but not arithmetic *capacity*. The qualitative analysis section is part of the result, not an afterthought.

## 1. Setup

Bitsandbytes is omitted: Kaggle's preinstalled `triton` is incompatible with `bitsandbytes==0.44.1`'s import path, and bf16 LoRA on a 360M model doesn't need 8-bit anyway. TRL is omitted in favor of `transformers.Trainer` + `peft` directly.

In [ ]:
!pip install -q -U transformers==4.45.2 peft==0.13.2 datasets==3.0.1 accelerate==1.0.1 anthropic
!pip uninstall -y bitsandbytes 2>/dev/null

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
os.environ["ANTHROPIC_API_KEY"] = user_secrets.get_secret("ANTHROPIC_API_KEY")
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

In [ ]:
import torch

assert torch.cuda.is_available(), "GPU not available — set Accelerator to GPU T4 in Kaggle settings"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 2. Configuration

All hyperparameters in one cell.

In [ ]:
# Model and data
BASE_MODEL = "HuggingFaceTB/SmolLM2-360M-Instruct"
TEACHER_MODEL = "claude-haiku-4-5"
N_TRAIN_PROBLEMS = 1500
N_HELD_OUT = 200

# LoRA — attention projections only
LORA_R = 16
LORA_ALPHA = 32
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Training
NUM_EPOCHS = 3
PER_DEVICE_BATCH = 4
GRAD_ACCUM_STEPS = 4              # effective batch = 16
LEARNING_RATE = 2e-4
MAX_SEQ_LEN = 1024

# Eval
EVAL_BATCH_SIZE = 8
EVAL_MAX_NEW_TOKENS = 512

SEED = 42

OUT_DIR = "/kaggle/working"
RAW_COTS_PATH = f"{OUT_DIR}/raw_cots.json"
FILTERED_COTS_PATH = f"{OUT_DIR}/filtered_cots.json"
ADAPTER_DIR = f"{OUT_DIR}/sft-final"

print(f"Base model:    {BASE_MODEL}")
print(f"Teacher model: {TEACHER_MODEL}")

## 3. Helpers

Answer extraction, gold parsing, and unit-aware comparison. Used in three places: filtering Claude's CoTs, grading the baseline, and grading the fine-tuned model. Defining once means all three use identical logic.

`answers_match` allows tolerance for unit conversions (cents↔dollars, minutes↔hours, etc.). Added after Claude was found reasoning in cents while GSM8K's gold was in dollars — both correct, but naive string equality marked them different.

In [ ]:
import re

def _normalize(s):
    s = s.replace(",", "").replace("$", "").strip()
    try:
        f = float(s)
        return str(int(f)) if f == int(f) else str(f)
    except ValueError:
        return s

def extract_answer(text):
    if text is None:
        return None
    m = re.search(r"\\boxed\{([^}]+)\}", text)
    if m: return _normalize(m.group(1))
    m = re.search(r"####\s*(-?[\d,]+\.?\d*)", text)
    if m: return _normalize(m.group(1))
    m = re.search(r"answer is\s*\$?(-?[\d,]+\.?\d*)", text, re.IGNORECASE)
    if m: return _normalize(m.group(1))
    nums = re.findall(r"-?[\d,]+\.?\d*", text)
    return _normalize(nums[-1]) if nums else None

def extract_gold(gsm8k_answer):
    return _normalize(gsm8k_answer.split("####")[-1].strip())

def answers_match(pred, gold):
    if pred is None or gold is None:
        return False
    try:
        p, g = float(pred), float(gold)
    except (ValueError, TypeError):
        return str(pred).strip() == str(gold).strip()
    if abs(p - g) < 1e-4:
        return True
    # cents/dollars, minutes/hours, hours/days, etc.
    for ratio in [100, 60, 24, 12, 1000, 7, 52, 365]:
        if g != 0 and abs(p / g - ratio) < 1e-4: return True
        if p != 0 and abs(g / p - ratio) < 1e-4: return True
    return False

assert extract_answer("...so #### 17") == "17"
assert extract_answer("\\boxed{3.5}") == "3.5"
assert answers_match("3", "300") is True
assert answers_match("3.0", "3") is True
assert answers_match("4", "5") is False

## 4. Generate teacher CoT data

Generate 1500 CoTs from Claude Haiku 4.5 on GSM8K training problems, then reject-sample by gold-answer match. If `filtered_cots.json` already exists, skip API generation.

Few-shot prompting was needed: the system prompt alone didn't prevent unit confusion. Three demonstrations (cents-to-dollars, minutes-to-hours, mixed) substantially reduced the failure rate. Examples beat instructions.

Cost: ~$2.70 (Haiku 4.5 at $1/$5 per M in/out, 1500 calls × ~80 in / 250 out tokens). Wall-clock ~45 min.

In [ ]:
import os
import json

if os.path.exists(FILTERED_COTS_PATH):
    with open(FILTERED_COTS_PATH) as f:
        filtered_cots = json.load(f)
    print(f"Loaded {len(filtered_cots)} cached filtered CoTs from {FILTERED_COTS_PATH}")
    SKIP_GENERATION = True
else:
    print("No cached CoTs found. Will generate from Claude (~$3, ~45 min).")
    SKIP_GENERATION = False

In [ ]:
if not SKIP_GENERATION:
    from datasets import load_dataset
    import random

    gsm8k = load_dataset("openai/gsm8k", "main")
    random.seed(7)
    train_problems = list(gsm8k["train"])
    random.shuffle(train_problems)
    to_label = train_problems[:N_TRAIN_PROBLEMS]
    print(f"Will label {len(to_label)} problems")

In [ ]:
if not SKIP_GENERATION:
    from anthropic import Anthropic
    import time
    from tqdm.auto import tqdm

    client = Anthropic()

    SYSTEM_PROMPT = """You are a math tutor. Solve the problem step by step.

Important reasoning guidelines:
- When a problem involves money, work in dollars throughout your reasoning, not cents.
  Convert cents to dollars early (e.g., "45 cents = $0.45") and keep all subsequent
  arithmetic in dollars.
- When a problem involves time, use the unit the question asks for (hours if it asks
  hours, minutes if minutes).
- The final answer should be in the most natural unit for the question.

End with the final answer in this exact format:
#### <number>

Use only the number itself after ####, with no units, no commas, no dollar signs."""

    FEW_SHOT = [
        {"role": "user", "content": "A pencil costs 30 cents. How much do 4 pencils cost?"},
        {"role": "assistant", "content": "I need to find the cost of 4 pencils.\n\nA pencil costs 30 cents, which is $0.30.\n\n4 pencils cost 4 × $0.30 = $1.20.\n\n#### 1.2"},
        {"role": "user", "content": "Two apples cost 50 cents. How much do 6 apples cost?"},
        {"role": "assistant", "content": "I need to find the cost of 6 apples.\n\nTwo apples cost 50 cents, which is $0.50. So one apple costs $0.50 / 2 = $0.25.\n\n6 apples cost 6 × $0.25 = $1.50.\n\n#### 1.5"},
        {"role": "user", "content": "A movie is 90 minutes long. How many hours is that?"},
        {"role": "assistant", "content": "I need to convert 90 minutes to hours.\n\nThere are 60 minutes in 1 hour.\n\n90 minutes ÷ 60 = 1.5 hours.\n\n#### 1.5"},
    ]

    def generate_cot(question, max_retries=3):
        messages = FEW_SHOT + [{"role": "user", "content": question}]
        for attempt in range(max_retries):
            try:
                resp = client.messages.create(
                    model=TEACHER_MODEL,
                    max_tokens=600,
                    system=SYSTEM_PROMPT,
                    messages=messages,
                    temperature=0.0,
                )
                return resp.content[0].text
            except Exception as e:
                print(f"Retry {attempt}: {e}")
                time.sleep(2 ** attempt)
        return None

    # Checkpoint every 100 in case Kaggle disconnects
    raw_cots = []
    for i, p in enumerate(tqdm(to_label, desc="Generating CoTs")):
        cot = generate_cot(p["question"])
        raw_cots.append({
            "question": p["question"],
            "gold": extract_gold(p["answer"]),
            "cot": cot,
        })
        if (i + 1) % 100 == 0:
            with open(RAW_COTS_PATH, "w") as f:
                json.dump(raw_cots, f)

    with open(RAW_COTS_PATH, "w") as f:
        json.dump(raw_cots, f)

    print(f"\nGenerated {len(raw_cots)} CoTs.")
    print(f"API failures: {sum(1 for x in raw_cots if x['cot'] is None)}")

In [ ]:
if not SKIP_GENERATION:
    filtered_cots = [
        x for x in raw_cots
        if x["cot"] is not None and answers_match(extract_answer(x["cot"]), x["gold"])
    ]

    with open(FILTERED_COTS_PATH, "w") as f:
        json.dump(filtered_cots, f)

    rate = len(filtered_cots) / len(raw_cots)
    print(f"Total CoTs:      {len(raw_cots)}")
    print(f"Correct CoTs:    {len(filtered_cots)}")
    print(f"Acceptance rate: {rate:.1%}")

print(f"{len(filtered_cots)} filtered CoTs available for training")

## 5. Format data for SFT

Wrap each (question, CoT) pair in SmolLM2's chat template, then tokenize. Training format must match inference format exactly.

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset
import random

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def format_example(item):
    messages = [
        {"role": "user", "content": item["question"]},
        {"role": "assistant", "content": item["cot"]},
    ]
    return {"text": tok.apply_chat_template(messages, tokenize=False)}

formatted = [format_example(x) for x in filtered_cots]

print("First formatted example (truncated):\n")
print(formatted[0]["text"][:800])
print(f"\n{len(formatted)} examples")

In [ ]:
# Train/eval split — eval set is for monitoring training, not the headline metric
random.seed(SEED)
shuffled = formatted.copy()
random.shuffle(shuffled)

sft_eval_examples = shuffled[:50]
sft_train_examples = shuffled[50:]

def tokenize_fn(examples):
    out = tok(examples["text"], truncation=True, max_length=MAX_SEQ_LEN, padding=False)
    out["labels"] = [ids.copy() for ids in out["input_ids"]]
    return out

sft_train = Dataset.from_list(sft_train_examples).map(
    tokenize_fn, batched=True, remove_columns=["text"]
)
sft_eval = Dataset.from_list(sft_eval_examples).map(
    tokenize_fn, batched=True, remove_columns=["text"]
)

print(f"Train: {len(sft_train)} examples")
print(f"Eval:  {len(sft_eval)} examples")

## 6. Eval harness

Greedy generation on a held-out test set, sampled from GSM8K's `test` split. Same harness used for baseline and fine-tuned eval. 200 problems gives ±~5pp 95% bootstrap CIs.

In [ ]:
from datasets import load_dataset
from tqdm.auto import tqdm

gsm8k = load_dataset("openai/gsm8k", "main")
random.seed(SEED)
test_problems = list(gsm8k["test"])
random.shuffle(test_problems)
held_out = test_problems[:N_HELD_OUT]
print(f"Held-out test set: {len(held_out)} problems")

# Left-padding for batched causal generation
tok.padding_side = "left"

def evaluate(model, tokenizer, problems, max_new_tokens=EVAL_MAX_NEW_TOKENS, batch_size=EVAL_BATCH_SIZE):
    correct = 0
    results = []
    for i in tqdm(range(0, len(problems), batch_size), desc="Evaluating"):
        batch = problems[i:i+batch_size]
        prompts = [
            tokenizer.apply_chat_template(
                [{"role": "user", "content": p["question"]}],
                tokenize=False, add_generation_prompt=True,
            )
            for p in batch
        ]
        inputs = tokenizer(
            prompts, return_tensors="pt", padding=True,
            truncation=True, max_length=MAX_SEQ_LEN,
        ).to(model.device)

        with torch.no_grad():
            outs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        for j, p in enumerate(batch):
            input_len = inputs.input_ids.shape[1]
            generated = tokenizer.decode(outs[j][input_len:], skip_special_tokens=True)
            pred = extract_answer(generated)
            gold = extract_gold(p["answer"])
            is_correct = answers_match(pred, gold)
            correct += is_correct
            results.append({
                "question": p["question"], "gold": gold, "pred": pred,
                "correct": is_correct, "raw_output": generated,
            })
    return correct / len(problems), results

## 7. Baseline eval

Untrained SmolLM2-360M on the held-out set. ~10 min on T4. Greedy decoding is deterministic.

In [ ]:
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

print(f"Model loaded on {model.device}")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

In [ ]:
baseline_acc, baseline_results = evaluate(model, tok, held_out)

print(f"\nBaseline accuracy: {baseline_acc:.3f} ({baseline_acc*100:.1f}%)")

with open(f"{OUT_DIR}/baseline_results.json", "w") as f:
    json.dump({
        "model": BASE_MODEL,
        "accuracy": baseline_acc,
        "n": len(held_out),
        "results": baseline_results,
    }, f)

print("\nSample baseline outputs:")
for r in baseline_results[:3]:
    status = "✓" if r["correct"] else "✗"
    print(f"\n{status} Gold: {r['gold']} | Pred: {r['pred']}")
    print(f"Output: {r['raw_output'][:250]}")

## 8. SFT-LoRA training

LoRA rank 16 over the four attention projections. Effective batch size 16 (per-device 4 × grad-accum 4), cosine LR with 5% warmup, 3 epochs. ~20 min on T4.

LoRA over full fine-tuning: ~1M trainable params instead of 360M, ~4× faster, negligible quality difference for narrow tasks. The adapter is also a 4MB artifact instead of a 720MB checkpoint.

In [ ]:
import gc

del model
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model

# Right-padding for training; left-padding only needed for batched generation
tok.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tok, padding=True, return_tensors="pt", pad_to_multiple_of=8,
)

training_args = TrainingArguments(
    output_dir=f"{OUT_DIR}/sft-out",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=True,
    report_to="none",
    save_total_limit=2,
    remove_unused_columns=False,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=sft_train,
    eval_dataset=sft_eval,
    tokenizer=tok,
    data_collator=data_collator,
)

trainer.train()
trainer.save_model(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}")

## 9. Fine-tuned eval

Merge LoRA adapter into base weights, then run the same eval harness.

In [ ]:
del trainer, model
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto",
)
ft_model = PeftModel.from_pretrained(base, ADAPTER_DIR)
ft_model = ft_model.merge_and_unload()
ft_model.eval()

tok.padding_side = "left"

print(f"Fine-tuned model on {ft_model.device}")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
ft_acc, ft_results = evaluate(ft_model, tok, held_out)

with open(f"{OUT_DIR}/baseline_results.json") as f:
    baseline = json.load(f)
baseline_acc = baseline["accuracy"]

print(f"\nBaseline:    {baseline_acc:.3f} ({baseline_acc*100:.1f}%)")
print(f"Fine-tuned:  {ft_acc:.3f} ({ft_acc*100:.1f}%)")
print(f"Delta:       +{(ft_acc-baseline_acc)*100:.1f}pp ({ft_acc/max(baseline_acc, 1e-9):.2f}x)")

with open(f"{OUT_DIR}/ft_results.json", "w") as f:
    json.dump({
        "model": f"{BASE_MODEL} + LoRA SFT (distilled from {TEACHER_MODEL})",
        "accuracy": ft_acc,
        "n": len(held_out),
        "baseline_accuracy": baseline_acc,
        "improvement_pp": (ft_acc - baseline_acc) * 100,
        "results": ft_results,
    }, f)

print("\nSample fine-tuned outputs:")
for r in ft_results[:3]:
    status = "✓" if r["correct"] else "✗"
    print(f"\n{status} Gold: {r['gold']} | Pred: {r['pred']}")
    print(f"Output: {r['raw_output'][:250]}")

## 10. Bootstrap confidence intervals

1. Marginal CIs: bootstrap baseline and fine-tuned accuracy independently.
2. Paired bootstrap on the per-problem difference, which controls for the fact that some held-out problems are intrinsically harder than others.

If the paired CI excludes zero, the improvement is significant.

In [ ]:
import numpy as np

def bootstrap_ci(results, n_boot=2000, alpha=0.05, seed=SEED):
    rng = np.random.default_rng(seed)
    correct = np.array([r["correct"] for r in results], dtype=int)
    n = len(correct)
    boots = np.array([correct[rng.integers(0, n, n)].mean() for _ in range(n_boot)])
    return np.percentile(boots, [100*alpha/2, 100*(1-alpha/2)])

baseline_ci = bootstrap_ci(baseline["results"])
ft_ci = bootstrap_ci(ft_results)

# Paired bootstrap on the per-problem difference
b_correct = np.array([r["correct"] for r in baseline["results"]], dtype=int)
f_correct = np.array([r["correct"] for r in ft_results], dtype=int)
diff = f_correct - b_correct
rng = np.random.default_rng(123)
diff_boots = np.array([diff[rng.integers(0, len(diff), len(diff))].mean() for _ in range(2000)])
diff_ci = np.percentile(diff_boots, [2.5, 97.5])
p_value = (diff_boots <= 0).mean()

print(f"Baseline:    {baseline['accuracy']*100:.1f}%  [95% CI: {baseline_ci[0]*100:.1f}, {baseline_ci[1]*100:.1f}]")
print(f"Fine-tuned:  {ft_acc*100:.1f}%  [95% CI: {ft_ci[0]*100:.1f}, {ft_ci[1]*100:.1f}]")
print(f"\nPaired difference: +{diff.mean()*100:.1f}pp  [95% CI: {diff_ci[0]*100:.1f}, {diff_ci[1]*100:.1f}]")
print(f"P(fine-tuned ≤ baseline) ≈ {p_value:.3f}")

with open(f"{OUT_DIR}/stats.json", "w") as f:
    json.dump({
        "baseline_acc": baseline["accuracy"], "baseline_ci": baseline_ci.tolist(),
        "ft_acc": ft_acc, "ft_ci": ft_ci.tolist(),
        "diff_mean": float(diff.mean()), "diff_ci": diff_ci.tolist(),
        "p_value": float(p_value),
    }, f)

## 11. Qualitative analysis: style vs. capability transfer

Headline accuracy is one signal. The other question: did the trained model adopt the teacher's reasoning structure? At 360M scale, distillation transfers reasoning *style* (format, structure) more readily than *capability* (the ability to do harder math).

In [ ]:
# Side-by-side: baseline vs. fine-tuned on the same problems
improved = [i for i in range(len(ft_results)) if ft_results[i]["correct"] and not baseline["results"][i]["correct"]]
regressed = [i for i in range(len(ft_results)) if baseline["results"][i]["correct"] and not ft_results[i]["correct"]]

print(f"Fine-tuned improved over baseline: {len(improved)}")
print(f"Fine-tuned regressed below baseline: {len(regressed)}")
print(f"Net: {len(improved) - len(regressed)} problems\n")

print("--- Wins (baseline wrong → fine-tuned correct) ---")
for idx in improved[:2]:
    b = baseline["results"][idx]
    f = ft_results[idx]
    print(f"\nQuestion: {b['question'][:200]}")
    print(f"Gold: {b['gold']}")
    print(f"\n  BASELINE  (✗ pred={b['pred']}): {b['raw_output'][:300]}...")
    print(f"\n  FINE-TUNED (✓ pred={f['pred']}): {f['raw_output'][:300]}...")
    print()

both_wrong = [i for i in range(len(ft_results))
              if not baseline["results"][i]["correct"] and not ft_results[i]["correct"]]

if both_wrong:
    print("\n--- Both wrong, but compare reasoning quality ---")
    idx = both_wrong[0]
    b = baseline["results"][idx]
    f = ft_results[idx]
    print(f"\nQuestion: {b['question'][:200]}")
    print(f"Gold: {b['gold']}")
    print(f"\n  BASELINE  (pred={b['pred']}): {b['raw_output'][:350]}...")
    print(f"\n  FINE-TUNED (pred={f['pred']}): {f['raw_output'][:350]}...")

In [ ]:
import statistics

baseline_lens = [len(r["raw_output"]) for r in baseline["results"]]
ft_lens = [len(r["raw_output"]) for r in ft_results]

baseline_with_marker = sum(1 for r in baseline["results"] if "####" in r["raw_output"])
ft_with_marker = sum(1 for r in ft_results if "####" in r["raw_output"])

print("Generation length (chars):")
print(f"  Baseline:   median={statistics.median(baseline_lens):.0f}, mean={statistics.mean(baseline_lens):.0f}")
print(f"  Fine-tuned: median={statistics.median(ft_lens):.0f}, mean={statistics.mean(ft_lens):.0f}")

print(f"\nFinal-answer marker (####) presence:")
print(f"  Baseline:   {baseline_with_marker}/{len(baseline['results'])} ({baseline_with_marker/len(baseline['results']):.1%})")
print(f"  Fine-tuned: {ft_with_marker}/{len(ft_results)} ({ft_with_marker/len(ft_results):.1%})")

## 12. Training loss curve

Train and eval loss across the 3 epochs.

In [ ]:
import matplotlib.pyplot as plt
import os

ckpts = sorted([d for d in os.listdir(f"{OUT_DIR}/sft-out") if d.startswith("checkpoint-")])
with open(f"{OUT_DIR}/sft-out/{ckpts[-1]}/trainer_state.json") as f:
    state = json.load(f)

train_loss = [(e["step"], e["loss"]) for e in state["log_history"] if "loss" in e]
eval_loss = [(e["step"], e["eval_loss"]) for e in state["log_history"] if "eval_loss" in e]

fig, ax = plt.subplots(figsize=(8, 5))
if train_loss:
    ax.plot([s for s,_ in train_loss], [l for _,l in train_loss], label="Train", alpha=0.7)
if eval_loss:
    ax.plot([s for s,_ in eval_loss], [l for _,l in eval_loss],
            label="Eval", marker="o", linewidth=2)
ax.set_xlabel("Step"); ax.set_ylabel("Loss"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title(f"SFT loss: {BASE_MODEL.split('/')[-1]} distilled from {TEACHER_MODEL}")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/loss_curve.png", dpi=120, bbox_inches="tight")
plt.show()

## 13. Push to HuggingFace Hub

In [ ]:
HF_REPO_NAME = "smollm2-360m-gsm8k-distilled-haiku"

ft_model.push_to_hub(HF_REPO_NAME)
tok.push_to_hub(HF_REPO_NAME)

## 14. Results summary

In [ ]:
import json

with open(f"{OUT_DIR}/baseline_results.json") as f:
    baseline = json.load(f)
with open(f"{OUT_DIR}/ft_results.json") as f:
    ft = json.load(f)
with open(f"{OUT_DIR}/stats.json") as f:
    stats = json.load(f)

print(f"CoT Distillation: {BASE_MODEL.split('/')[-1]} ← {TEACHER_MODEL}\n")
print(f"  Training data:    {len(filtered_cots)} CoTs (from {N_TRAIN_PROBLEMS} generated)")
print(f"  Held-out test:    {N_HELD_OUT} GSM8K problems")
print(f"  LoRA config:      r={LORA_R}, alpha={LORA_ALPHA}")
print(f"  Training:         {NUM_EPOCHS} epochs, lr={LEARNING_RATE}, eff. batch=16")
print()
print(f"  Baseline:    {stats['baseline_acc']*100:.1f}%  "
      f"[95% CI: {stats['baseline_ci'][0]*100:.1f}–{stats['baseline_ci'][1]*100:.1f}]")
print(f"  Fine-tuned:  {stats['ft_acc']*100:.1f}%  "
      f"[95% CI: {stats['ft_ci'][0]*100:.1f}–{stats['ft_ci'][1]*100:.1f}]")
print(f"  Δ (paired):  +{stats['diff_mean']*100:.1f}pp  "
      f"[95% CI: {stats['diff_ci'][0]*100:.1f}–{stats['diff_ci'][1]*100:.1f}]")
print(f"  P(no improvement) ≈ {stats['p_value']:.3f}")